# Next step: step 9b, every output gets a catalog
**What it changes:** with an output, every run is recorded in a catalog (its models, configs, statuses, and later its
scores), whether or not it saves data. `run.save` now only controls the model folders and arrays. The output can also
be a `.json` file: a catalog on its own.

| `output` | What's written |
|---|---|
| `runs/` (no extension), no `run.save` | `runs/catalog.json` only |
| `runs/`, with `run.save` | `runs/catalog.json`, plus model folders with `model.json` and `.npy` files |
| `gen_07.json` | That file: a catalog alone. With `run.save`, an error (saving data needs a folder) |
| any other extension (`runs.txt`, `runs.v2/`) | An error |

Folder or file is told apart by the name alone, so it works before anything exists. A `.json` catalog collects runs
just like a folder's: models matched, runs appended. This replaces "a run that only reads writes nothing". It's what the
command line (Step 11) builds on: there an output will be required, and `run.save` optional.

## What changed
**`src/galactic_tidal_constraints/simr/simr.py`.** The output's two parts, from its name:
```python
def output_paths(output):
    output = Path(output)
    if output.suffix == ".json":
        return output, None                          # a catalog file, no folder for data
    if output.suffix == "":
        return output / "catalog.json", output       # a folder
    raise ValueError(f"the output must be a folder (no extension) or a .json catalog file: {output}")
```
In `run`, the catalog and its record now depend on having an output, not on `run.save`:
```python
catalog_file, root = output_paths(output) if output is not None else (None, None)
if saved and root is None:
    raise ValueError(f"run.save names data to save, which needs an output folder, but {where}")
catalog = _open_catalog(catalog_file) if catalog_file is not None else None
ids = _model_ids(catalog, models) if catalog is not None else [None] * len(models)
if catalog is not None:
    ...   # the run record, written at the start and completed at the end, as before
```
Every model a run uses is now added to the catalog (so `result.ids` has no `None` with an output).

**Tests:** `tests/test_simr.py`, section "Step 9b". **Docs:** `docs/design/simr.md`, The output.
**Decision:** `PLAN.md`, Structure (Output catalog; the output is a folder or a `.json` file).

In [ ]:
import tempfile
from pathlib import Path

import numpy as np

from galactic_tidal_constraints.simr import simr
from galactic_tidal_constraints.simr import simulation as sim

params = np.array([1.08119, -1.66798, -2.06826, -0.18438, -0.39693, -0.02919, 0.51654, 0.36192, 0.94043, 0.43877,
                   12.97415, 135.11902, 97.54035, 305.01225])
heavier = params.copy()
heavier[sim.PARAM_INDEX["mass2"]] *= 2
models = np.stack([params, heavier])
request = {"run": {"return": ["sim.timing"]}, "simulator": [{"n1": 500, "n2": 500}]}
work = Path(tempfile.mkdtemp(prefix="simr_9b_"))


def show(folder):
    for path in sorted(folder.rglob("*")):
        print("  ", path.relative_to(folder))

## 1. A folder, nothing saved: just the catalog

In [ ]:
result = simr.run(models, request, output=work / "runs")
show(work)
[record] = simr.load_catalog(work / "runs")["runs"]
print("ids:", result.ids, "  status:", record["status"], "  save:", record["run"]["save"])

## 2. A `.json` file: a catalog alone, collecting runs

In [ ]:
simr.run(models, request, output=work / "gen_01.json")
simr.run(models[::-1], request, output=work / "gen_01.json")
show(work)
catalog = simr.load_catalog(work / "gen_01.json")
print("models:", list(catalog["models"]), "  runs:", [run["model_ids"] for run in catalog["runs"]])

## 3. What's an error

In [ ]:
attempts = {
    "save with a .json output": lambda: simr.run(models, {**request, "run": {"save": ["sim.orbit"]}}, output=work / "gen_02.json"),
    "a .txt output": lambda: simr.run(models, request, output=work / "runs.txt"),
    "a folder with a dot": lambda: simr.run(models, request, output=work / "runs.v2"),
}
for label, call in attempts.items():
    try:
        call()
    except ValueError as error:
        print(f"{label:26s} -> {error}")
show(work)